In [2]:
import pandas as pd

In [3]:
raw= pd.read_csv("teaching_prices_messy.csv")

In [4]:
raw

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
...,...,...,...,...,...,...
92,2026-03-19,Pulse C,City 1,207.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
93,2026-03-19,Pulse C,City 2,213.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
94,2026-03-19,Sweetener D,City 1,178.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
95,2026-03-19,Sweetener D,City 2,184.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


# 2_work.ipynb — Cleaning the Dirty Ledger

**Assignment 2 · The Prices Desk**  
Working on a **copy** of `teaching_prices_messy.csv`.  
All decisions follow the diagnosis recorded in `1_attempt.md`.

## 1. Load original data (never modify the source file)

In [5]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("teaching_prices_messy.csv")

print("Shape:", df_raw.shape)
print("\nDtypes:")
print(df_raw.dtypes)
print("\nFirst 8 rows:")
df_raw.head(8)

Shape: (97, 6)

Dtypes:
date          object
item          object
city          object
price         object
unit          object
data_label    object
dtype: object

First 8 rows:


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 2. Before-cleaning snapshot

In [6]:
print("=== BEFORE CLEANING ===")
print("Rows:", len(df_raw))
print("Missing prices:", df_raw['price'].isna().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
print("\nItem value counts:")
print(df_raw['item'].value_counts())
print("\nPrice sample (raw):")
print(df_raw['price'].head(10).tolist())
print("\nAny price containing comma:")
print(df_raw[df_raw['price'].astype(str).str.contains(',', na=False)][['date','item','city','price']])
print("\nExtreme values (>1000 after naive coerce):")
tmp = pd.to_numeric(df_raw['price'].astype(str).str.replace(',',''), errors='coerce')
print(df_raw.loc[tmp > 1000, ['date','item','city','price']])

=== BEFORE CLEANING ===
Rows: 97
Missing prices: 1
Duplicate rows: 1

Item value counts:
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64

Price sample (raw):
['101.0', '107.0', '298.0', '304.0', '200.0', nan, '149.0', '155.0', '99.5', '105.5']

Any price containing comma:
          date      item    city  price
17  2026-01-15  Staple A  City 2  1,250

Extreme values (>1000 after naive coerce):
          date         item    city  price
17  2026-01-15     Staple A  City 2  1,250
30  2026-01-22  Sweetener D  City 1   9999


## 3. Cleaning steps (on a copy)

Each step corresponds to a planned decision from `1_attempt.md`.

In [7]:
# Work on a copy
df = df_raw.copy()

# --- Step A: remove exact duplicates ---
print("Rows before drop_duplicates:", len(df))
df = df.drop_duplicates()
print("Rows after drop_duplicates:", len(df))
# Expected: 96

Rows before drop_duplicates: 97
Rows after drop_duplicates: 96


In [8]:
# --- Step B: fix the mislabeled item "oil b" → "Staple A" ---
# IMPORTANT: use exact match so we do not touch the real "Oil B" rows
mask = df['item'] == 'oil b'
print("Rows that will be relabelled:")
print(df.loc[mask, ['date','item','city','price']])
df.loc[mask, 'item'] = 'Staple A'
print("\nItem value counts after fix:")
print(df['item'].value_counts())

Rows that will be relabelled:
          date   item    city  price
40  2026-02-05  oil b  City 1  108.5

Item value counts after fix:
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64


In [9]:
# --- Step C: clean price strings and convert to numeric ---
df['price'] = (
    df['price']
    .astype(str)
    .str.replace(',', '', regex=False)
    .replace({'nan': pd.NA, 'None': pd.NA, '': pd.NA})
)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print("Price dtype now:", df['price'].dtype)
print("Missing after conversion:", df['price'].isna().sum())
print("Describe:")
print(df['price'].describe())

Price dtype now: float64
Missing after conversion: 1
Describe:
count      95.000000
mean      317.731579
std      1012.659212
min        99.500000
25%       151.000000
50%       200.000000
75%       304.000000
max      9999.000000
Name: price, dtype: float64


In [10]:
# --- Step D: null the extreme outliers (> 1000) ---
outlier_mask = df['price'] > 1000
print("Outliers that will be set to NaN:")
print(df.loc[outlier_mask, ['date','item','city','price']])
df.loc[outlier_mask, 'price'] = pd.NA
print("\nMissing prices now:", df['price'].isna().sum())

Outliers that will be set to NaN:
          date         item    city   price
17  2026-01-15     Staple A  City 2  1250.0
30  2026-01-22  Sweetener D  City 1  9999.0

Missing prices now: 3


In [11]:
# --- Step E: parse dates ---
df['date'] = pd.to_datetime(df['date'])
print("Date dtype:", df['date'].dtype)
print("Date range:", df['date'].min(), "→", df['date'].max())
print("Unique weeks:", df['date'].nunique())

Date dtype: datetime64[ns]
Date range: 2026-01-01 00:00:00 → 2026-03-19 00:00:00
Unique weeks: 12


In [12]:
print("=== AFTER CLEANING ===")
print("Final shape:", df.shape)
print("Missing prices:", df['price'].isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nItem value counts (should be 4 items only):")
print(df['item'].value_counts())
print("\nDtypes:")
print(df.dtypes)
print("\nPrice describe (clean):")
print(df['price'].describe())
print("\nSample of cleaned data:")
df.head(10)

=== AFTER CLEANING ===
Final shape: (96, 6)
Missing prices: 3
Duplicate rows: 0

Item value counts (should be 4 items only):
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64

Dtypes:
date          datetime64[ns]
item                  object
city                  object
price                float64
unit                  object
data_label            object
dtype: object

Price describe (clean):
count     93.000000
mean     203.607527
std       79.922070
min       99.500000
25%      149.000000
50%      200.000000
75%      298.000000
max      350.000000
Name: price, dtype: float64

Sample of cleaned data:


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
9,2026-01-08,Staple A,City 2,105.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [13]:
# Quick panel completeness check
panel = df.groupby(['date', 'item', 'city']).size().unstack(fill_value=0)
print("Observations per date × item × city:")
print(panel)

Observations per date × item × city:
city                    City 1  City 2
date       item                       
2026-01-01 Oil B             1       1
           Pulse C           1       1
           Staple A          1       1
           Sweetener D       1       1
2026-01-08 Oil B             1       1
           Pulse C           1       1
           Staple A          1       1
           Sweetener D       1       1
2026-01-15 Oil B             1       1
           Pulse C           1       1
           Staple A          1       1
           Sweetener D       1       1
2026-01-22 Oil B             1       1
           Pulse C           1       1
           Staple A          1       1
           Sweetener D       1       1
2026-01-29 Oil B             1       1
           Pulse C           1       1
           Staple A          1       1
           Sweetener D       1       1
2026-02-05 Oil B             1       1
           Pulse C           1       1
           Staple A        

## 5. Summary of changes

| Action | Rows affected | Result |
|--------|---------------|--------|
| Drop exact duplicate | 1 | 97 → 96 rows |
| Relabel "oil b" → Staple A | 1 | consistent 4-item panel |
| Strip comma + to_numeric | all | price is float64 |
| Null extreme outliers (9999 & 1250) | 2 | 3 missing prices total |
| Parse dates | all | datetime64 |

The cleaned table is honest: the three missing prices are visible and documented.